# SASRec Continuous Time-Aware BPI2012 Colab Train (`refine_ml50_do035` baseline)

Colab notebook for the continuous/log-delta time-aware SASRec experiment using `refine_ml50_do035` as the fixed baseline.

Goals:
- reuse the completed `refine_ml50_do035` baseline results
- train only the new continuous time-aware runs
- compare baseline vs `continuous` time encoding
- evaluate under both `NDCG@10` and `NDCG@5` model-selection criteria


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
BASELINE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
TIMEAWARE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg10'
TIMEAWARE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('BASELINE_NDCG5_OUTPUT_DIR:', BASELINE_NDCG5_OUTPUT_DIR)
print('TIMEAWARE_NDCG10_OUTPUT_DIR:', TIMEAWARE_NDCG10_OUTPUT_DIR)
print('TIMEAWARE_NDCG5_OUTPUT_DIR:', TIMEAWARE_NDCG5_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
BASELINE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5
TIMEAWARE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg10
TIMEAWARE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg5


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASELINE_NDCG5_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG10_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG5_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


In [8]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Experiment design

Fixed baseline setting:
- `refine_ml50_do035`
- `hidden_units=50, num_blocks=2, num_heads=1, maxlen=50, lr=0.001, dropout=0.35`
- seeds: `42`, `2024`, `7`

Comparison targets:
- baseline (reuse existing completed runs)
- time-aware `continuous/log-delta`

Time-aware design:
- `x = item_embedding + positional_embedding + time_embedding`
- time source: `delta_prev_seconds`
- time encoding: `[log1p(delta_prev_seconds), is_first_event] -> Linear(2, hidden_units)`


## Check existing baseline runs

These baseline runs should already exist and must not be retrained.


In [10]:
from pathlib import Path

baseline_ndcg10_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
baseline_ndcg5_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

for label, output_dir, run_names in [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR), baseline_ndcg10_runs),
    ('Baseline NDCG@5', Path(BASELINE_NDCG5_OUTPUT_DIR), baseline_ndcg5_runs),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Baseline NDCG@10
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS
Baseline NDCG@5
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS


## Check planned continuous runs

Only train runs that are still missing.


In [11]:
planned_ndcg10 = [
    'timeaware_conti_refine_ml50_do035_s42',
    'timeaware_conti_refine_ml50_do035_s2024',
    'timeaware_conti_refine_ml50_do035_s7',
]
planned_ndcg5 = [
    'timeaware_conti_refine_ml50_do035_s42',
    'timeaware_conti_refine_ml50_do035_s2024',
    'timeaware_conti_refine_ml50_do035_s7',
]

for label, output_dir, run_names in [
    ('Continuous NDCG@10', Path(TIMEAWARE_NDCG10_OUTPUT_DIR), planned_ndcg10),
    ('Continuous NDCG@5', Path(TIMEAWARE_NDCG5_OUTPUT_DIR), planned_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Continuous NDCG@10
timeaware_conti_refine_ml50_do035_s42 OK
timeaware_conti_refine_ml50_do035_s2024 OK
timeaware_conti_refine_ml50_do035_s7 OK
Continuous NDCG@5
timeaware_conti_refine_ml50_do035_s42 OK
timeaware_conti_refine_ml50_do035_s2024 OK
timeaware_conti_refine_ml50_do035_s7 OK


## Train continuous time-aware runs for `NDCG@10`

Run these cells only if the corresponding run directory does not already exist.


### timeaware_conti_refine_ml50_do035_s42


In [12]:
!python src/train_sasrec.py \
  --run_name timeaware_conti_refine_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg10/timeaware_conti_refine_ml50_do035_s42
epoch=1, loss=0.6162
epoch=2, loss=0.2829
epoch=3, loss=0.2114
epoch=4, loss=0.1745
epoch=5, loss=0.1501
valid [full], NDCG@5: 0.6166, HR@5: 0.7019, NDCG@10: 0.6822, HR@10: 0.9171, MRR: 0.6211
valid [sampled], NDCG@5: 0.5460, HR@5: 0.5482, NDCG@10: 0.5555, HR@10: 0.5788, MRR: 0.5611
test [full], NDCG@5: 0.6189, HR@5: 0.7512, NDCG@10: 0.6578, HR@10: 0.8828, MRR: 0.5960
test [sampled], NDCG@5: 0.2139, HR@5: 0.2196, NDCG@10: 0.2613, HR@10: 0.3728, MRR: 0.2602
saved eval checkpoint: /content/drive/MyDr

### timeaware_conti_refine_ml50_do035_s2024


In [13]:
!python src/train_sasrec.py \
  --run_name timeaware_conti_refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg10/timeaware_conti_refine_ml50_do035_s2024
epoch=1, loss=0.6394
epoch=2, loss=0.2888
epoch=3, loss=0.2095
epoch=4, loss=0.1695
epoch=5, loss=0.1449
valid [full], NDCG@5: 0.6595, HR@5: 0.7914, NDCG@10: 0.7272, HR@10: 0.9930, MRR: 0.6474
valid [sampled], NDCG@5: 0.5548, HR@5: 0.5619, NDCG@10: 0.5669, HR@10: 0.6003, MRR: 0.5715
test [full], NDCG@5: 0.7886, HR@5: 0.9749, NDCG@10: 0.7974, HR@10: 1.0000, MRR: 0.7327
test [sampled], NDCG@5: 0.2162, HR@5: 0.2785, NDCG@10: 0.2735, HR@10: 0.4546, MRR: 0.2488
saved eval checkpoint: /content/drive/My

### timeaware_conti_refine_ml50_do035_s7


In [14]:
!python src/train_sasrec.py \
  --run_name timeaware_conti_refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg10/timeaware_conti_refine_ml50_do035_s7
epoch=1, loss=0.6094
epoch=2, loss=0.2957
epoch=3, loss=0.2156
epoch=4, loss=0.1749
epoch=5, loss=0.1504
valid [full], NDCG@5: 0.6836, HR@5: 0.8441, NDCG@10: 0.7274, HR@10: 0.9780, MRR: 0.6526
valid [sampled], NDCG@5: 0.5638, HR@5: 0.5646, NDCG@10: 0.5699, HR@10: 0.5842, MRR: 0.5823
test [full], NDCG@5: 0.6657, HR@5: 0.8923, NDCG@10: 0.6890, HR@10: 0.9626, MRR: 0.6041
test [sampled], NDCG@5: 0.2493, HR@5: 0.2523, NDCG@10: 0.2738, HR@10: 0.3321, MRR: 0.2932
saved eval checkpoint: /content/drive/MyDri

## Train continuous time-aware runs for `NDCG@5`

Run these cells only if the corresponding run directory does not already exist.


### timeaware_conti_refine_ml50_do035_s42


In [15]:
!python src/train_sasrec.py \
  --run_name timeaware_conti_refine_ml50_do035_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg5/timeaware_conti_refine_ml50_do035_s42
epoch=1, loss=0.6162
epoch=2, loss=0.2829
epoch=3, loss=0.2114
epoch=4, loss=0.1745
epoch=5, loss=0.1501
valid [full], NDCG@5: 0.6166, HR@5: 0.7019, NDCG@10: 0.6822, HR@10: 0.9171, MRR: 0.6211
valid [sampled], NDCG@5: 0.5460, HR@5: 0.5482, NDCG@10: 0.5555, HR@10: 0.5788, MRR: 0.5611
test [full], NDCG@5: 0.6189, HR@5: 0.7512, NDCG@10: 0.6578, HR@10: 0.8828, MRR: 0.5960
test [sampled], NDCG@5: 0.2139, HR@5: 0.2196, NDCG@10: 0.2613, HR@10: 0.3728, MRR: 0.2602
saved eval checkpoint: /content/drive/MyDriv

### timeaware_conti_refine_ml50_do035_s2024


In [16]:
!python src/train_sasrec.py \
  --run_name timeaware_conti_refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg5/timeaware_conti_refine_ml50_do035_s2024
epoch=1, loss=0.6394
epoch=2, loss=0.2888
epoch=3, loss=0.2095
epoch=4, loss=0.1695
epoch=5, loss=0.1449
valid [full], NDCG@5: 0.6595, HR@5: 0.7914, NDCG@10: 0.7272, HR@10: 0.9930, MRR: 0.6474
valid [sampled], NDCG@5: 0.5548, HR@5: 0.5619, NDCG@10: 0.5669, HR@10: 0.6003, MRR: 0.5715
test [full], NDCG@5: 0.7886, HR@5: 0.9749, NDCG@10: 0.7974, HR@10: 1.0000, MRR: 0.7327
test [sampled], NDCG@5: 0.2162, HR@5: 0.2785, NDCG@10: 0.2735, HR@10: 0.4546, MRR: 0.2488
saved eval checkpoint: /content/drive/MyDr

### timeaware_conti_refine_ml50_do035_s7


In [17]:
!python src/train_sasrec.py \
  --run_name timeaware_conti_refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_encoding continuous \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_continuous_refine_ml50_do035_ndcg5/timeaware_conti_refine_ml50_do035_s7
epoch=1, loss=0.6094
epoch=2, loss=0.2957
epoch=3, loss=0.2156
epoch=4, loss=0.1749
epoch=5, loss=0.1504
valid [full], NDCG@5: 0.6836, HR@5: 0.8441, NDCG@10: 0.7274, HR@10: 0.9780, MRR: 0.6526
valid [sampled], NDCG@5: 0.5638, HR@5: 0.5646, NDCG@10: 0.5699, HR@10: 0.5842, MRR: 0.5823
test [full], NDCG@5: 0.6657, HR@5: 0.8923, NDCG@10: 0.6890, HR@10: 0.9626, MRR: 0.6041
test [sampled], NDCG@5: 0.2493, HR@5: 0.2523, NDCG@10: 0.2738, HR@10: 0.3321, MRR: 0.2932
saved eval checkpoint: /content/drive/MyDrive

## Rebuild result tables from run folders

This avoids schema issues and lets us combine existing baseline runs with new continuous runs safely.


In [6]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()

    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue

        summary_path = run_dir / "metrics_summary.json"
        config_path = run_dir / "config.json"
        if not summary_path.exists() or not config_path.exists():
            continue

        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        config = json.loads(config_path.read_text(encoding="utf-8"))

        row = {
            "run_name": summary.get("run_name"),
            "run_dir": str(run_dir),
            "completed_at": summary.get("completed_at"),
            "best_epoch": summary.get("best_epoch"),
            "checkpoint_best": summary.get("checkpoint_best"),
            "checkpoint_last": summary.get("checkpoint_last"),
            "metrics_history": summary.get("metrics_history"),
            "config_path": str(config_path),
            "metrics_summary": str(summary_path),
            "maxlen": config.get("maxlen"),
            "dropout_rate": config.get("dropout_rate"),
            "hidden_units": config.get("hidden_units"),
            "seed": config.get("seed"),
            "selection_metric": config.get("selection_metric"),
            "use_time_embedding": config.get("use_time_embedding", False),
            "time_encoding": config.get("time_encoding"),
            "time_feature_dim": config.get("time_feature_dim"),
        }

        best_valid = summary.get("best_valid", {})
        best_test = summary.get("best_test_at_best_valid", {})

        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)

        row.update({
            "best_valid_full_ndcg@10": pick(best_valid, "full", "ndcg@10"),
            "best_valid_full_hr@10": pick(best_valid, "full", "hr@10"),
            "best_valid_full_ndcg@5": pick(best_valid, "full", "ndcg@5"),
            "best_valid_full_hr@5": pick(best_valid, "full", "hr@5"),
            "best_valid_full_mrr": pick(best_valid, "full", "mrr"),

            "best_test_full_ndcg@10": pick(best_test, "full", "ndcg@10"),
            "best_test_full_hr@10": pick(best_test, "full", "hr@10"),
            "best_test_full_ndcg@5": pick(best_test, "full", "ndcg@5"),
            "best_test_full_hr@5": pick(best_test, "full", "hr@5"),
            "best_test_full_mrr": pick(best_test, "full", "mrr"),

            "best_valid_sampled_ndcg@10": pick(best_valid, "sampled", "ndcg@10"),
            "best_valid_sampled_hr@10": pick(best_valid, "sampled", "hr@10"),
            "best_valid_sampled_ndcg@5": pick(best_valid, "sampled", "ndcg@5"),
            "best_valid_sampled_hr@5": pick(best_valid, "sampled", "hr@5"),
            "best_valid_sampled_mrr": pick(best_valid, "sampled", "mrr"),

            "best_test_sampled_ndcg@10": pick(best_test, "sampled", "ndcg@10"),
            "best_test_sampled_hr@10": pick(best_test, "sampled", "hr@10"),
            "best_test_sampled_ndcg@5": pick(best_test, "sampled", "ndcg@5"),
            "best_test_sampled_hr@5": pick(best_test, "sampled", "hr@5"),
            "best_test_sampled_mrr": pick(best_test, "sampled", "mrr"),
        })

        rows.append(row)

    return pd.DataFrame(rows)


In [7]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

## NDCG@10 comparison summary


In [8]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
timeaware_runs = [
    'timeaware_conti_refine_ml50_do035_s42',
    'timeaware_conti_refine_ml50_do035_s2024',
    'timeaware_conti_refine_ml50_do035_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG10_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['time_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['time_variant'] = 'continuous'

df_ndcg10 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg10 = df_ndcg10.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'time_variant', 'use_time_embedding', 'time_encoding', 'time_feature_dim',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,time_variant,use_time_embedding,time_encoding,time_feature_dim,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,refine_ml50_do035_s7,7,baseline,False,None,None,0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.0,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750
1,refine_ml50_do035_s42,42,baseline,False,None,None,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.0,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
2,refine_ml50_do035_s2024,2024,baseline,False,None,None,0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.0,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183
3,timeaware_conti_refine_ml50_do035_s7,7,continuous,True,continuous,2,0.742856,0.957962,0.709444,0.849013,0.679322,0.919483,1.0,0.919392,0.999729,0.892217,0.595109,0.674728,0.565226,0.580299,0.585420,0.450081,0.644405,0.389823,0.458819,0.414366
4,timeaware_conti_refine_ml50_do035_s42,42,continuous,True,continuous,2,0.735692,0.969771,0.684508,0.810492,0.666913,0.671893,1.0,0.666020,0.983502,0.561349,0.581198,0.640022,0.558554,0.568380,0.577778,0.115456,0.241412,0.062731,0.070733,0.127650
5,timeaware_conti_refine_ml50_do035_s2024,2024,continuous,True,continuous,2,0.758122,0.974855,0.737981,0.908341,0.691520,0.830097,1.0,0.829480,0.998243,0.772035,0.595450,0.679092,0.563381,0.577391,0.587073,0.396034,0.527692,0.344150,0.362081,0.388127


In [9]:
summary_ndcg10 = df_ndcg10.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg10


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10      best_test_full_ndcg@5           best_test_full_hr@5          best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean  std                  mean       std                mean      std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
time_variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              
baseline                    0.735259  0.006374              0.977329  0.015618               0.702263  0.009451             0.872599  0.029271            0.662771  0.002053               0.846684  0.062093                  1.0  0.0              0.818405  0.078916            0.911416  0.05528           0.798923  0.080599                   0.572973  0.004280                 0.604843  0.007136                  0.561597  0.005491                0.569159  0.005422               0.579235  0.005635                  0.372338  0.102528                0.576445  0.125558                 0.309277  0.098875               0.383139  0.105721              0.333054  0.093175
continuous                  0.745557  0.011456              0.967529  0.008667               0.710644  0.026757             0.855949  0.049292            0.679252  0.012303               0.807158  0.125379                  1.0  0.0              0.804964  0.128453            0.993825  0.00897           0.741867  0.167484                   0.590585  0.008132                 0.664614  0.021409                  0.562387  0.003445                0.575356  0.006215               0.583424  0.004959                  0.320524  0.179638                0.471170  0.207357                 0.265568  0.177140               0.297211  0.202012              0.310048  0.158505

Interpretation guide for NDCG@10:
- compare `best_valid_full_ndcg@10` and `best_test_full_ndcg@10` first
- then check whether sampled metrics and MRR show a similar trend
- baseline is reused; only continuous time-aware runs are newly trained here


## NDCG@5 comparison summary


In [10]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
timeaware_runs = [
    'timeaware_conti_refine_ml50_do035_s42',
    'timeaware_conti_refine_ml50_do035_s2024',
    'timeaware_conti_refine_ml50_do035_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG5_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG5_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['time_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['time_variant'] = 'continuous'

df_ndcg5 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg5 = df_ndcg5.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'time_variant', 'use_time_embedding', 'time_encoding', 'time_feature_dim',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,time_variant,use_time_embedding,time_encoding,time_feature_dim,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,refine_ml50_do035_s7,7,baseline,False,None,None,0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.0,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750
1,refine_ml50_do035_s42,42,baseline,False,None,None,0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.0,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
2,refine_ml50_do035_s2024,2024,baseline,False,None,None,0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.0,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183
3,timeaware_conti_refine_ml50_do035_s7,7,continuous,True,continuous,2,0.742856,0.957962,0.709444,0.849013,0.679322,0.919483,1.0,0.919392,0.999729,0.892217,0.595109,0.674728,0.565226,0.580299,0.585420,0.450081,0.644405,0.389823,0.458819,0.414366
4,timeaware_conti_refine_ml50_do035_s42,42,continuous,True,continuous,2,0.734529,0.972588,0.695617,0.847062,0.664040,0.673986,1.0,0.669747,0.988100,0.563469,0.590503,0.645318,0.570988,0.584380,0.587901,0.157738,0.303520,0.098997,0.115672,0.158573
5,timeaware_conti_refine_ml50_do035_s2024,2024,continuous,True,continuous,2,0.758122,0.974855,0.737981,0.908341,0.691520,0.830097,1.0,0.829480,0.998243,0.772035,0.595450,0.679092,0.563381,0.577391,0.587073,0.396034,0.527692,0.344150,0.362081,0.388127


In [11]:
summary_ndcg5 = df_ndcg5.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg5


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10      best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean  std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
time_variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               
baseline                    0.735259  0.006374              0.977329  0.015618               0.702263  0.009451             0.872599  0.029271            0.662771  0.002053               0.846684  0.062093                  1.0  0.0              0.818405  0.078916            0.911416  0.055280           0.798923  0.080599                   0.572973  0.004280                 0.604843  0.007136                  0.561597  0.005491                0.569159  0.005422               0.579235  0.005635                  0.372338  0.102528                0.576445  0.125558                 0.309277  0.098875               0.383139  0.105721              0.333054  0.093175
continuous                  0.745169  0.011965              0.968468  0.009169               0.714347  0.021603             0.868139  0.034830            0.678294  0.013769               0.807855  0.124251                  1.0  0.0              0.806206  0.126440            0.995357  0.006329           0.742574  0.166343                   0.593687  0.002763                 0.666380  0.018370                  0.566532  0.003968                0.580690  0.003511               0.586798  0.001263                  0.334618  0.155548                0.491872  0.173242                 0.277656  0.156400               0.312190  0.176930              0.320355  0.140721

Interpretation guide for NDCG@5:
- compare `best_valid_full_ndcg@5` and `best_test_full_ndcg@5` first
- then check whether sampled metrics and MRR show a similar trend
- baseline is reused; only continuous time-aware runs are newly trained here
